In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


df = pd.read_csv('../data/processed/salary_benchmark_engineered.csv')
print(f"Dataset loaded: {df.shape}")

Dataset loaded: (5500, 19)


In [2]:
df.columns

Index(['age', 'gender', 'education_level', 'job_title', 'seniority_level',
       'years_experience', 'years_at_company', 'industry', 'company_size',
       'location', 'performance_rating', 'num_promotions',
       'num_certifications', 'remote_ratio', 'col_index', 'base_salary',
       'log_base_salary', 'sqrt_years_experience', 'sqrt_years_at_company'],
      dtype='str')

In [3]:
target = 'log_base_salary'
target_raw = 'base_salary'

cat_cols = ['gender', 'job_title', 'industry', 'location']

#drop originals and focus on engineered features
num_cols = df.drop(columns=cat_cols + [
    'base_salary', 'log_base_salary', 
    'seniority_level', 'education_level', 'company_size'
]).columns.tolist()

X = df[cat_cols + num_cols]
y_log = df[target]
y_raw = df[target_raw]

X_train, X_test, y_log_train, y_log_test, y_raw_train, y_raw_test = train_test_split(
    X, y_log, y_raw, test_size=0.2, random_state=42
)

print(f"Training features: {X_train.shape[1]}")


Training features: 14


In [ ]:
naive_prediction = np.full_like(y_raw_test, y_raw_train.mean())

baseline_rmse = np.sqrt(mean_squared_error(y_raw_test, naive_prediction))
baseline_mae = mean_absolute_error(y_raw_test, naive_prediction)

print("Naive Baseline Mean Prediction")
print(f"RMSE: ${baseline_rmse:,.2f}")
print(f"MAE:  ${baseline_mae:,.2f}")

Naive Baseline Mean Prediction
RMSE: $43,276.88
MAE:  $34,170.41


In [12]:
from sklearn.impute import SimpleImputer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ])

# Quick check to ensure it fits properly
X_train_preprocessed = preprocessor.fit_transform(X_train)
num_transformer = Pipeline([
('imputer', SimpleImputer(strategy='median')),
('scaler', StandardScaler())
])

cat_transformer = Pipeline([
('imputer', SimpleImputer(strategy='most_frequent')),
('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
transformers=[
('num', num_transformer, num_cols),
('cat', cat_transformer, cat_cols)
]
)

In [ ]:

def evaluate_model(model, name):
    model.fit(X_train, y_log_train)
    preds_log = model.predict(X_test)
    preds_raw = np.exp(preds_log)
    
    rmse = np.sqrt(mean_squared_error(y_raw_test, preds_raw))
    mae = mean_absolute_error(y_raw_test, preds_raw)
    r2 = r2_score(y_raw_test, preds_raw)
    
    print(f"--- {name} ---")
    print(f"RMSE: ${rmse:,.2f}")
    print(f"MAE:  ${mae:,.2f}")
    print(f"R²:   {r2:.4f}\n")
    
    return model, preds_raw

In [13]:
ridge_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RidgeCV(cv=5))
])

lasso_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LassoCV(cv=5, max_iter=10000, random_state=42))
])

fitted_ridge, ridge_preds = evaluate_model(ridge_pipe, "Ridge Regression")
fitted_lasso, lasso_preds = evaluate_model(lasso_pipe, "Lasso Regression")

--- Ridge Regression ---
RMSE: $28,180.18
MAE:  $22,138.27
R²:   0.5755

--- Lasso Regression ---
RMSE: $28,104.90
MAE:  $22,086.38
R²:   0.5778



In [8]:
missing = X.isna().sum()
print("Total NaNs in X:", int(missing.sum()))
print("\nTop columns with NaNs:")
print(missing[missing > 0].sort_values(ascending=False).head(20))
print("\nCategorical NaNs:")
print(X[['gender','job_title','industry','location']].isna().sum())

Total NaNs in X: 1100

Top columns with NaNs:
performance_rating       330
years_experience         275
sqrt_years_experience    275
num_certifications       220
dtype: int64

Categorical NaNs:
gender       0
job_title    0
industry     0
location     0
dtype: int64
